# AIラジオ大学｜ワンクリック音声化（Irodori-TTS・無料）

Googleドライブの **制作ボード** で「音声化待ち」になっている講義を、無料の音声合成 **Irodori-TTS** でMP3にして、ドライブの **「AIラジオ大学 制作室 / 音声」** フォルダに保存します。保存後、ボードの状態は「登録待ち」に進み、会議ログにも記録されます。

**使い方**
1. 上のメニュー「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選んで保存（無料）
2. 下の ①〜④ の ▶ を、上から順に押す（①は初回5〜10分かかります）
3. 途中でGoogleアカウントの許可画面が出たら「許可」を押す

最初は「一度に作る本数」を **1** にして、声の感じを確かめるのがおすすめです。


In [ ]:
#@title ① 準備（初回のみ・5〜10分）
import subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし（ランタイムをT4 GPUにしてください）')
!pip -q install uv
![ -d /content/Irodori-TTS ] || git clone -q https://github.com/Aratako/Irodori-TTS.git /content/Irodori-TTS
!cd /content/Irodori-TTS && uv sync -q --extra cu128
!wget -q -O /content/Irodori-TTS/irodori_worker.py https://raw.githubusercontent.com/bananaraw123/ai-radio-uni/main/tools/irodori_worker.py
print('準備完了')

In [ ]:
#@title ② 原稿を読み込む
一度に作る本数 = 1  #@param {type:"integer"}
import json, re, shutil, google.auth, gspread
from pathlib import Path
from google.colab import auth
from googleapiclient.discovery import build
auth.authenticate_user()
creds, _ = google.auth.default()
drive, docs, gc = build('drive', 'v3', credentials=creds), build('docs', 'v1', credentials=creds), gspread.authorize(creds)

def find(name, mime):
    r = drive.files().list(q=f"name = '{name}' and mimeType = '{mime}' and trashed = false", fields='files(id)').execute()['files']
    if not r: raise SystemExit(f'「{name}」がドライブに見つかりません')
    return r[0]['id']
FOLDER = find('AIラジオ大学 制作室', 'application/vnd.google-apps.folder')
SHEET = find('制作ボード｜AIラジオ大学', 'application/vnd.google-apps.spreadsheet')
ws = gc.open_by_key(SHEET).sheet1
rows = ws.get_all_values()
links = ws.get('E1:E200', value_render_option='FORMULA')
head = rows[0]; C = {h: i for i, h in enumerate(head)}

# 教授ごとの声（自由に書き換えてOK）
声 = {
 '人工知能学部': '落ち着いた中年の男性。大学の講義で、明快に、穏やかに語りかけている。',
 '宇宙学部': 'やわらかく優しい声の女性。夜のラジオ番組のように、静かに語りかけている。',
 '哲学部': '低めの声でゆっくり話す年配の男性。思慮深く、穏やかに講義している。',
 '心理学部': '明るく親しみやすい女性。聞き手に笑顔で語りかけるように話している。',
 '神秘学部': '低く静かな声の男性。神秘的な雰囲気で、ささやくように語っている。',
 '宗教学部': '穏やかで品のある年配の男性。落ち着いた口調で、ゆっくりと語りかけている。',
 '経済学部': 'テンポのよい中年の男性。少しユーモラスに、楽しそうに講義している。',
 '歴史学部': '温かい声の男性。語り部のように、物語を聞かせるように話している。',
}
MARK = 'ここから下が読み上げる本文'
jobs = Path('/content/jobs'); shutil.rmtree(jobs, ignore_errors=True); jobs.mkdir()
Path('/content/out').mkdir(exist_ok=True)
todo = []
for r, row in enumerate(rows[1:], start=2):
    if row[C['状態']] != '音声化待ち' or len(todo) >= 一度に作る本数: continue
    m = re.search(r'/d/([\w-]+)', str(links[r - 1][0]) if r - 1 < len(links) and links[r - 1] else '')
    if not m: print('原稿リンクが読めません:', row[C['タイトル']]); continue
    text = drive.files().export(fileId=m.group(1), mimeType='text/plain').execute().decode('utf-8-sig')
    body = text.split(MARK, 1)[1].split('\n', 1)[1] if MARK in text else text
    out = f"/content/out/{row[C['MP3ファイル名']]}"
    (jobs / f'{r:03d}.json').write_text(json.dumps({'text': body.strip(), 'caption': 声.get(row[C['学部']], '落ち着いた大人の話者。大学の講義のように語りかけている。'), 'out_mp3': out, 'seed': 1000 + r}, ensure_ascii=False))
    todo.append((r, row[C['タイトル']], out))
for r, t, _ in todo: print(f'音声化します：{t}')
if not todo: print('「音声化待ち」の講義はありません')

In [ ]:
#@title ③ 音声を作る（1本あたり数分〜）
!cd /content/Irodori-TTS && uv run --no-sync python irodori_worker.py /content/jobs

In [ ]:
#@title ④ ドライブに保存して記録
import datetime, os
from googleapiclient.http import MediaFileUpload
r = drive.files().list(q=f"name = '音声' and '{FOLDER}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false", fields='files(id)').execute()['files']
VOICE = r[0]['id'] if r else drive.files().create(body={'name': '音声', 'parents': [FOLDER], 'mimeType': 'application/vnd.google-apps.folder'}, fields='id').execute()['id']
done = []
for row, title, out in todo:
    if not os.path.exists(out): print('できていません：', title); continue
    drive.files().create(body={'name': os.path.basename(out), 'parents': [VOICE]}, media_body=MediaFileUpload(out, mimetype='audio/mpeg'), fields='id').execute()
    ws.update_cell(row, C['状態'] + 1, '登録待ち'); ws.update_cell(row, C['Muse音声'] + 1, 'Irodoriで生成')
    done.append(title); print('保存しました：', title)
if done:
    ROOM = find('00 AI会議室｜はじめに読む', 'application/vnd.google-apps.document')
    line = f"\n{datetime.date.today()} Irodori（Colab）：{len(done)}本を音声化し、制作室の「音声」フォルダに保存しました（{'、'.join(done)}）。状態は「登録待ち」です。"
    docs.documents().batchUpdate(documentId=ROOM, body={'requests': [{'insertText': {'endOfSegmentLocation': {}, 'text': line}}]}).execute()
    print('ボードと会議ログを更新しました。スマホのアプリで「研究室 → MP3を追加」から登録してください。')